In [1]:
import flowArcFirstTry as eh
import json
import pandas as pd
import numpy as np
from copy import deepcopy

In [2]:
file_path = 'italien/group_3/gruppe_3_italien_10.json.json'
df = pd.read_json(file_path)

# Auswahl und Anordnung der Spalten mit den Nullen an den exakten Positionen
instance_patients = np.array([
    [
        row['patient_id'],
        row['arrival_time'],
        row['realized_processing_time'],
        0, 0,row['arrival_time']+row['max_wait_time'], 0, 
        row['weight'],
        row['expected_processing_time']
    ]
    for _, row in df.iterrows()
], dtype=int)

# Ergebnis anzeigen
print(instance_patients)

[[  0 480  12   0   0 540   0   2  17]
 [  1 482  17   0   0 542   0   2  17]
 [  2 484  17   0   0 499   0   3  21]
 [  3 487   5   0   0 547   0   2  17]
 [  4 488   6   0   0 548   0   2  17]
 [  5 498  14   0   0 618   0   1  13]
 [  6 498  22   0   0 513   0   3  21]
 [  7 500  13   0   0 560   0   2  17]
 [  8 501  26   0   0 561   0   2  17]
 [  9 503  13   0   0 623   0   1  13]
 [ 10 505  13   0   0 565   0   2  17]
 [ 11 506  10   0   0 566   0   2  17]
 [ 12 508   4   0   0 568   0   2  17]
 [ 13 516  19   0   0 531   0   3  21]
 [ 14 519  18   0   0 579   0   2  17]
 [ 15 522  35   0   0 537   0   3  21]
 [ 16 526  16   0   0 586   0   2  17]
 [ 17 528  18   0   0 588   0   2  17]
 [ 18 529  19   0   0 649   0   1  13]
 [ 19 531  22   0   0 591   0   2  17]
 [ 20 534  18   0   0 654   0   1  13]
 [ 21 534  24   0   0 549   0   3  21]
 [ 22 538   2   0   0 553   0   3  21]
 [ 23 538  29   0   0 553   0   3  21]
 [ 24 540  23   0   0 600   0   2  17]
 [ 25 549   5   0   0 669

In [3]:


instance_patients

array([[  0, 480,  12,   0,   0, 540,   0,   2,  17],
       [  1, 482,  17,   0,   0, 542,   0,   2,  17],
       [  2, 484,  17,   0,   0, 499,   0,   3,  21],
       [  3, 487,   5,   0,   0, 547,   0,   2,  17],
       [  4, 488,   6,   0,   0, 548,   0,   2,  17],
       [  5, 498,  14,   0,   0, 618,   0,   1,  13],
       [  6, 498,  22,   0,   0, 513,   0,   3,  21],
       [  7, 500,  13,   0,   0, 560,   0,   2,  17],
       [  8, 501,  26,   0,   0, 561,   0,   2,  17],
       [  9, 503,  13,   0,   0, 623,   0,   1,  13],
       [ 10, 505,  13,   0,   0, 565,   0,   2,  17],
       [ 11, 506,  10,   0,   0, 566,   0,   2,  17],
       [ 12, 508,   4,   0,   0, 568,   0,   2,  17],
       [ 13, 516,  19,   0,   0, 531,   0,   3,  21],
       [ 14, 519,  18,   0,   0, 579,   0,   2,  17],
       [ 15, 522,  35,   0,   0, 537,   0,   3,  21],
       [ 16, 526,  16,   0,   0, 586,   0,   2,  17],
       [ 17, 528,  18,   0,   0, 588,   0,   2,  17],
       [ 18, 529,  19,   0, 

In [4]:
import Neighbourhoods as nh

def add_patient(waiting_room_schedule, doctor, patient):
    waiting_room_schedule[doctor] = np.vstack([waiting_room_schedule[doctor], patient])

def call_next_patient(waiting_room_schedule, doctor):
    if waiting_room_schedule[doctor].shape[0] > 0:  # Überprüfen, ob das Array nicht leer ist
        patient = waiting_room_schedule[doctor][0]
        delete_patient_form_waiting_room(waiting_room_schedule, doctor)
        return patient
    else:
        return np.array([])

def delete_patient_form_waiting_room(waiting_room_schedule, doctor):
    waiting_room_schedule[doctor] = waiting_room_schedule[doctor][1:]

def start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end , doctor):
    patient = call_next_patient(waiting_room_schedule, doctor)
    if patient.shape[0]>0:
        estimated_treatment_end[doctor-1] = time + patient[8]
        treatment_end[doctor-1] = time+patient[2]
        #print(f"{time}: Beginn Treatment for {patient}")
        schedule_treated_patient(schedule, doctor, patient)
        nh.calculate_start_and_endtimes_per_doc(schedule, doctor)

def calculate_completion_time_per_doc(waiting_room_schedule, doctor_completion, estimated_treatment_end, doctor, deterministic=True):
    calculate_waiting_room_values_per_doc(waiting_room_schedule, doctor, estimated_treatment_end, deterministic=deterministic)
    doctor_completion[doctor-1] = waiting_room_schedule[doctor][-1, 4]

def schedule_treated_patient(schedule, doctor, patient):
    add_patient(schedule, doctor, patient)

def greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient, deterministic=True):
    next_available_doctor = np.argmin(doctor_completion) + 1
    doctor_completion[next_available_doctor-1] += patient[8]
    add_patient(waiting_room_schedule, next_available_doctor, patient)
    reprioritize_patients_by_urgency_level(waiting_room_schedule, next_available_doctor)
    estimated_treatment_end_doctor = estimated_treatment_end[next_available_doctor-1]
    calculate_completion_time_per_doc(waiting_room_schedule, doctor_completion, estimated_treatment_end_doctor, next_available_doctor, deterministic=deterministic)
    
def reprioritize_patients_by_urgency_level(waiting_room_schedule, doctor):
    sorted_patients_indices = np.lexsort((
    waiting_room_schedule[doctor][:, 5],  # Sortierung nach due date = 5, nach ankunftszeit = 1
    -waiting_room_schedule[doctor][:, 7]   # Sortierung nach Urgency weight
    #waiting_room_schedule[doctor][:,5] # Sortierung nach due date
))
    #sorted_patients_indices = np.argsort(waiting_room_schedule[doctor][:, 7])[::-1]
    waiting_room_schedule[doctor] = waiting_room_schedule[doctor][sorted_patients_indices]

def dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient_list, deterministic=True):
    incoming_patient_priority_by_weights_indices = np.argsort(patient_list[:, 7])[::-1]
    incoming_patient_priority_by_weights = patient_list[incoming_patient_priority_by_weights_indices]

    for patient in incoming_patient_priority_by_weights:
        #print(f"{patient}")
        greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient, deterministic=deterministic)

    # for patient in patient_list:
    #     greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient)

def calculate_waiting_room_values_per_doc(schedule, key, start_time=0,deterministic=True):
    if deterministic:
        nh.calculate_start_and_endtimes_per_doc(schedule, key, start_time)
    else:
        nh.calculate_start_and_endtimes_per_doc_non_det(schedule, key, start_time)
    

In [ ]:
# Waiting_room
doctor_count = 1
doctor_completion = np.zeros(doctor_count)

waiting_room_schedule = {}
waiting_room_weighted_tardiness = 0
schedule = {}
deterministic = False
metaheuristic = "VNS"
#metaheuristic = 0
#metaheuristic = "SA"

for i in range(1, doctor_count+1):
    waiting_room_schedule[i] = np.array([], dtype=int).reshape(0, 9)

schedule = deepcopy(waiting_room_schedule)

start_time = instance_patients[0][1]
end_time = instance_patients[-1][1]

#start_time = 486
#end_time = 550

estimated_treatment_end = np.full(doctor_count, start_time)
treatment_end = estimated_treatment_end.copy()
#new_patient = False
time = start_time

while time <= end_time:
    arrived_patients = instance_patients[:, 1] == time

    arrived_patients_list = instance_patients[arrived_patients]
    if arrived_patients_list.shape[0] > 0:
        #new_patient = True
        #print(f"{time}: Patient arrives:")
        dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, arrived_patients_list, deterministic=deterministic)
        waiting_room_weighted_tardiness = 0
        for doctor in waiting_room_schedule:
            waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        

        # Metaheuristik
        if metaheuristic == "VNS" and waiting_room_weighted_tardiness != 0:
            new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=10, time_limit=60, start_time_array=estimated_treatment_end, deterministic=deterministic)
            if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
                print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}")
                waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
            else:
                print(f"{time}: VNS {waiting_room_weighted_tardiness} Keine Verbesserung auf {new_waiting_room_weighted_tardiness}")
        
        #new_patient = False, 
        # waiting_room_weighted_tardiness = 0
        # for doctor in waiting_room_schedule:
        #     waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        #print(f"Time {time} waiting room tardiness = {waiting_room_weighted_tardiness}")
    
    #feasible_metaheuristic = any(array.size > 1 for array in waiting_room_schedule.values())

        

    free_doctors = np.where(treatment_end <= time)[0]
    free_doctors += 1

    if len(free_doctors)>0:
        for doctor in free_doctors:
            start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end, doctor)
            
            #print(f"estimated_end: {estimated_treatment_end}, treatment_end: {treatment_end}, completion_time: {doctor_completion}")
            #for doctor in waiting_room_schedule:
                #print(f"Waiting_room_schedule: {waiting_room_schedule[doctor]}")
            #print(f"Schedule:")
            #print(schedule)
            if metaheuristic == "SA" and waiting_room_weighted_tardiness != 0:
                #print(f"{time} SA triggered with weighted tardiness: {waiting_room_weighted_tardiness}\n{waiting_room_schedule}")
                new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.simulated_annealing(waiting_room_schedule, waiting_room_weighted_tardiness, start_time_array=estimated_treatment_end, start_temperature=10, Imax=2000, time_limit=10, Max_d=1200, deterministic=deterministic)
                print(f"SA Done")
                if waiting_room_weighted_tardiness > new_waiting_room_weighted_tardiness:
                    print(f"{time}: SA {waiting_room_weighted_tardiness} auf {new_waiting_room_weighted_tardiness}")
                    waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                    #waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
        # for doctor in waiting_room_schedule:
        #     waiting_room_weighted_tardiness = 0
        #     waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)

    #print(f"{time}: completion_time: {doctor_completion}, estimated_end: {estimated_treatment_end} and end: {treatment_end}")
    #for doctor in waiting_room_schedule:
        #print(f"{doctor}:\n {waiting_room_schedule[doctor]}")
        #print(f"{doctor} schedule :\n {schedule[doctor]}")
    waiting_room_weighted_tardiness = 0
    for doctor in waiting_room_schedule:
        waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        
    time += 1
    waiting_room_empy = all(array.size == 0 for array in waiting_room_schedule.values())
    if not waiting_room_empy:
        end_time += 1
    
    # if arrived_patients_list.shape[0] > 0:
    #     #print(f" Patient: {arrived_patients_list} \n estimated_end: {estimated_treatment_end}, end: {treatment_end}; completion_time: {doctor_completion}")
    #     for doctor in waiting_room_schedule:
    #        #print(f"doctor {doctor} waiting_room: \n {waiting_room_schedule[doctor]} \n")
    #        #print(f"doctor {doctor} schedule: \n {schedule[doctor]} \n")

schedule_weighted_tardiness = 0
for doctor in schedule:
    schedule_weighted_tardiness += nh.weighted_tardiness_per_doc(schedule, doctor)

print(f"{schedule}, {schedule_weighted_tardiness}")

488: VNS 8 Keine Verbesserung auf 8
498: VNS 48 Keine Verbesserung auf 48
500: VNS 98 Keine Verbesserung auf 98
501: VNS 181 Keine Verbesserung auf 181
503: VNS 190 Keine Verbesserung auf 190
505: VNS 332 Keine Verbesserung auf 332
506: VNS 506 Keine Verbesserung auf 506
508: VNS 710 Keine Verbesserung auf 710
516: VNS 1000 Keine Verbesserung auf 1000
519: VNS 1250 Keine Verbesserung auf 1250
522: VNS 1712 Keine Verbesserung auf 1712
526: VNS 2024 Keine Verbesserung auf 2024
528: VNS 2366 Keine Verbesserung auf 2366
529: VNS 2502 Keine Verbesserung auf 2502
531: VNS 2889 Keine Verbesserung auf 2889
534: VNS 3741 Keine Verbesserung auf 3741
538: VNS 5226 Keine Verbesserung auf 5226
540: VNS 5774 Keine Verbesserung auf 5774
549: VNS 6014 Keine Verbesserung auf 6014
555: VNS 6732 Keine Verbesserung auf 6732
560: VNS 7329 Keine Verbesserung auf 7329
574: VNS 7593 Keine Verbesserung auf 7593
580: VNS 7864 Keine Verbesserung auf 7864
581: VNS 8487 Keine Verbesserung auf 8487
583: VNS 9435 Ke

In [ ]:
schedule

{1: array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  3, 486,  19, 498, 517, 606,   0,   1,  19],
        [  5, 495,  28, 517, 545, 555,   0,   2,   5],
        [  9, 507,   9, 545, 554, 522,  23,   3,  30],
        [  8, 500,  16, 554, 570, 560,   0,   2,   1],
        [ 16, 540,  17, 570, 587, 555,  15,   3,  17],
        [ 12, 524,  15, 587, 602, 584,   3,   2,   6],
        [ 10, 513,  13, 602, 615, 573,  29,   2,  19],
        [ 15, 535,  24, 615, 639, 595,  20,   2,  21],
        [ 20, 540,  16, 639, 655, 600,  39,   2,  17],
        [ 13, 526,  13, 655, 668, 586,  69,   2,  23],
        [ 23, 556,  19, 668, 687, 571,  97,   3,  40],
        [ 26, 557,  21, 687, 708, 572, 115,   3,  29],
        [ 24, 557,  24, 708, 732, 617,  91,   2,  19],
        [ 30, 569,  18, 732, 750, 584, 148,   3,  16],
        [ 32, 581,  17, 750, 767, 596, 154,   3,  21],
        [ 34, 603,  11, 767, 778, 618, 149,   3,  15],
        [ 35, 605,  18, 778, 796, 665, 113,   2,  10],
       

In [ ]:
schedule
nh.weighted_tardiness_per_doc(schedule, 1) + nh.weighted_tardiness_per_doc(schedule, 2)

13051

In [ ]:
sched_det = deepcopy(schedule)
sched_non_det = deepcopy(schedule)


In [ ]:
def calculate_waiting_room_values_per_doc(schedule, key, start_time=0, deterministic=True):
    for i in range(len(schedule[key])):
        if i == 0 and start_time == 0:
            schedule[key][0, [4]] = schedule[key][0, [1]] + schedule[key][0, [2]]
            schedule[key][0, [3]] = schedule[key][0, [1]]
        elif i == 0 and start_time > 0:
            schedule[key][0, [4]] = start_time + schedule[key][0, [8]]
            schedule[key][0, [3]] = start_time
        elif deterministic:
            schedule[key][i, [3]] = max(schedule[key][i-1, [4]], schedule[key][i, [1]])
            schedule[key][i, [4]] = schedule[key][i, [3]] + schedule[key][i, [2]]
        else:
            schedule[key][i, [3]] = max(schedule[key][i-1, [4]], schedule[key][i, [1]])
            schedule[key][i, [4]] = schedule[key][i, [3]] + schedule[key][i, [8]]

    # Berechne das Maximum aus 0 und der Differenz der 4. und 6. Spalte
    result = np.maximum(0, schedule[key][:, 3] - schedule[key][:, 5])

    # Weisen Sie das Ergebnis der 7. Spalte zu
    schedule[key][:, 6] = result    

In [6]:
file_path = '../hongkong.json'
df = pd.read_json(file_path)

# Auswahl und Anordnung der Spalten mit den Nullen an den exakten Positionen
instance_patients = np.array([
    [
        row['patient_id'],
        row['arrival_time'],
        row['realized_processing_time'],
        0, 0,row['arrival_time']+row['max_wait_time'], 0, 
        row['weight'],
        row['expected_processing_time']
    ]
    for _, row in df.iterrows()
], dtype=int)

# Ergebnis anzeigen
print(instance_patients)

[[  1 484  26 ...   0   2  23]
 [  2 485  17 ...   0   1  11]
 [  3 488  23 ...   0   2  17]
 ...
 [138 714  29 ...   0   1  11]
 [139 714  22 ...   0   1   6]
 [140 715  15 ...   0   1  15]]


In [30]:
# Waiting_room
doctor_count = 8
doctor_completion = np.zeros(doctor_count)

waiting_room_schedule = {}
waiting_room_weighted_tardiness = 0
schedule = {}
deterministic = False
metaheuristic = "VNS"
#metaheuristic = 0
#metaheuristic = "SA"

for i in range(1, doctor_count+1):
    waiting_room_schedule[i] = np.array([], dtype=int).reshape(0, 9)

schedule = deepcopy(waiting_room_schedule)

start_time = instance_patients[0][1]
end_time = instance_patients[-1][1]

#start_time = 486
#end_time = 550

estimated_treatment_end = np.full(doctor_count, start_time)
treatment_end = estimated_treatment_end.copy()
#new_patient = False
time = start_time

while time <= end_time:
    arrived_patients = instance_patients[:, 1] == time

    arrived_patients_list = instance_patients[arrived_patients]
    if arrived_patients_list.shape[0] > 0:
        #new_patient = True
        #print(f"{time}: Patient arrives:")
        dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, arrived_patients_list, deterministic=deterministic)
        waiting_room_weighted_tardiness = 0
        for doctor in waiting_room_schedule:
            waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        

        # Metaheuristik
        if metaheuristic == "VNS" and waiting_room_weighted_tardiness != 0:
            new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=3, time_limit=20, start_time_array=estimated_treatment_end, deterministic=deterministic)
            if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
                print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}, {len(waiting_room_schedule[1])*doctor_count}")
                waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
            else:
                print(f"{time}: VNS {waiting_room_weighted_tardiness} Keine Verbesserung auf {new_waiting_room_weighted_tardiness}, {len(waiting_room_schedule[1])*doctor_count}")
                print(f"\n waiting room: {waiting_room_schedule}")
        
        #new_patient = False, 
        # waiting_room_weighted_tardiness = 0
        # for doctor in waiting_room_schedule:
        #     waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        #print(f"Time {time} waiting room tardiness = {waiting_room_weighted_tardiness}")
    
    #feasible_metaheuristic = any(array.size > 1 for array in waiting_room_schedule.values())

    free_doctors = np.where(treatment_end <= time)[0]
    free_doctors += 1

    if len(free_doctors)>0:
        for doctor in free_doctors:
            start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end, doctor)
            
            #print(f"estimated_end: {estimated_treatment_end}, treatment_end: {treatment_end}, completion_time: {doctor_completion}")
            #for doctor in waiting_room_schedule:
                #print(f"Waiting_room_schedule: {waiting_room_schedule[doctor]}")
            #print(f"Schedule:")
            #print(schedule)
            if metaheuristic == "SA" and waiting_room_weighted_tardiness != 0:
                #print(f"{time} SA triggered with weighted tardiness: {waiting_room_weighted_tardiness}\n{waiting_room_schedule}")
                new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.simulated_annealing(waiting_room_schedule, waiting_room_weighted_tardiness, start_time_array=estimated_treatment_end, start_temperature=10, Imax=2000, time_limit=20, Max_d=120, deterministic=deterministic)
                print(f"SA Done")
                if waiting_room_weighted_tardiness > new_waiting_room_weighted_tardiness:
                    print(f"{time}: SA {waiting_room_weighted_tardiness} auf {new_waiting_room_weighted_tardiness}")
                    waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                    #waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
        # for doctor in waiting_room_schedule:
        #     waiting_room_weighted_tardiness = 0
        #     waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)

    #print(f"{time}: completion_time: {doctor_completion}, estimated_end: {estimated_treatment_end} and end: {treatment_end}")
    #for doctor in waiting_room_schedule:
        #print(f"{doctor}:\n {waiting_room_schedule[doctor]}")
        #print(f"{doctor} schedule :\n {schedule[doctor]}")
    waiting_room_weighted_tardiness = 0
    for doctor in waiting_room_schedule:
        waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        
    time += 1
    waiting_room_empy = all(array.size == 0 for array in waiting_room_schedule.values())
    if not waiting_room_empy:
        end_time += 1
    
    # if arrived_patients_list.shape[0] > 0:
    #     #print(f" Patient: {arrived_patients_list} \n estimated_end: {estimated_treatment_end}, end: {treatment_end}; completion_time: {doctor_completion}")
    #     for doctor in waiting_room_schedule:
    #        #print(f"doctor {doctor} waiting_room: \n {waiting_room_schedule[doctor]} \n")
    #        #print(f"doctor {doctor} schedule: \n {schedule[doctor]} \n")

schedule_weighted_tardiness = 0
for doctor in schedule:
    schedule_weighted_tardiness += nh.weighted_tardiness_per_doc(schedule, doctor)

print(f"{schedule}, {schedule_weighted_tardiness}")

702: VNS 13 Verbesserung auf 0, 32
{1: array([[  1, 484,  26, 484, 510, 544,   0,   2,  23],
       [ 15, 523,  15, 523, 538, 643,   0,   1,  11],
       [ 19, 535,  28, 538, 566, 655,   0,   1,  14],
       [ 43, 563,   4, 566, 570, 623,   0,   2,   2],
       [ 26, 540,  22, 570, 592, 660,   0,   1,  16],
       [ 38, 556,  28, 592, 620, 676,   0,   1,  12],
       [ 44, 564,  27, 620, 647, 684,   0,   1,  10],
       [ 85, 621,  14, 647, 661, 681,   0,   2,  16],
       [ 47, 567,  26, 661, 687, 687,   0,   1,   6],
       [118, 684,   8, 687, 695, 744,   0,   2,  10],
       [ 60, 583,  23, 695, 718, 703,   0,   1,  17],
       [ 71, 598,  19, 718, 737, 718,   0,   1,  10],
       [ 80, 608,  13, 737, 750, 728,   9,   1,   9],
       [103, 661,  32, 750, 782, 781,   0,   1,  11],
       [121, 691,  23, 782, 805, 811,   0,   1,  10],
       [134, 709,  28, 805, 833, 829,   0,   1,  15],
       [138, 714,  29, 833, 862, 834,   0,   1,  11]]), 2: array([[  2, 485,  17, 485, 502, 605, 

In [ ]:
import numpy as np

schedule =  {1: np.array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  3, 486,  19, 498, 517, 606,   0,   1,  19],
        [  5, 495,  28, 517, 545, 555,   0,   2,   5],
        [  9, 507,   9, 545, 554, 522,  23,   3,  30],
        [  8, 500,  16, 554, 570, 560,   0,   2,   1],
        [ 16, 540,  17, 570, 587, 555,  15,   3,  17],
        [ 12, 524,  15, 587, 602, 584,   3,   2,   6],
        [ 10, 513,  13, 602, 615, 573,  29,   2,  19],
        [ 15, 535,  24, 615, 639, 595,  20,   2,  21],
        [ 20, 540,  16, 639, 655, 600,  39,   2,  17],
        [ 13, 526,  13, 655, 668, 586,  69,   2,  23],
        [ 23, 556,  19, 668, 687, 571,  97,   3,  40],
        [ 26, 557,  21, 687, 708, 572, 115,   3,  29],
        [ 24, 557,  24, 708, 732, 617,  91,   2,  19],
        [ 30, 569,  18, 732, 750, 584, 148,   3,  16],
        [ 32, 581,  17, 750, 767, 596, 154,   3,  21],
        [ 34, 603,  11, 767, 778, 618, 149,   3,  15],
        [ 35, 605,  18, 778, 796, 665, 113,   2,  10],
        [ 37, 622,  23, 796, 819, 637, 159,   3,   2],
        [ 38, 628,  10, 819, 829, 643, 176,   3,  15],
        [ 40, 634,  25, 829, 854, 694, 135,   2,  18],
        [ 42, 642,  25, 854, 879, 657, 197,   3,  31],
        [ 45, 661,  20, 879, 899, 676, 203,   3,  13],
        [ 46, 666,  14, 899, 913, 726, 173,   2,  22],
        [ 50, 692,  24, 913, 937, 752, 161,   2,  11],
        [ 51, 702,  11, 937, 948, 822, 115,   1,   9],
        [ 52, 705,  25, 948, 973, 765, 183,   2,  19],
        [ 53, 713,  10, 973, 983, 728, 245,   3,  26],
        [ 56, 718,  14, 983, 997, 778, 205,   2,  11]]),
 2: np.array([[  2, 486,  22, 486, 508, 546,   0,   2,  20],
        [  4, 488,  15, 508, 523, 548,   0,   2,  15],
        [ 11, 518,   9, 523, 532, 533,   0,   3,  28],
        [  7, 499,  19, 532, 551, 559,   0,   2,  13],
        [ 19, 540,  14, 551, 565, 555,   0,   3,  15],
        [  6, 497,  23, 565, 588, 557,   8,   2,  20],
        [ 14, 527,  24, 588, 612, 587,   1,   2,  18],
        [ 21, 540,  20, 612, 632, 600,  12,   2,  14],
        [ 17, 540,  11, 632, 643, 600,  32,   2,  18],
        [ 18, 540,  15, 643, 658, 600,  43,   2,  24],
        [ 22, 549,  10, 658, 668, 609,  49,   2,  22],
        [ 27, 557,  24, 668, 692, 572,  96,   3,  26],
        [ 25, 557,  11, 692, 703, 617,  75,   2,  20],
        [ 28, 566,  20, 703, 723, 626,  77,   2,  11],
        [ 29, 567,  14, 723, 737, 627,  96,   2,  22],
        [ 31, 578,  10, 737, 747, 593, 144,   3,  29],
        [ 33, 587,  19, 747, 766, 647, 100,   2,  25],
        [ 36, 605,  12, 766, 778, 665, 101,   2,  23],
        [ 39, 633,  24, 778, 802, 693,  85,   2,  12],
        [ 41, 642,  22, 802, 824, 702, 100,   2,  20],
        [ 43, 644,  10, 824, 834, 659, 165,   3,  24],
        [ 44, 661,  18, 834, 852, 721, 113,   2,  24],
        [ 47, 669,  19, 852, 871, 729, 123,   2,   7],
        [ 48, 674,  15, 871, 886, 689, 182,   3,  10],
        [ 49, 678,  18, 886, 904, 693, 193,   3,  23],
        [ 54, 716,   5, 904, 909, 731, 173,   3,  16],
        [ 55, 716,  18, 909, 927, 776, 133,   2,  15],
        [ 57, 719,  14, 927, 941, 779, 148,   2,  13]])}

doctor_load = np.array([0, 0]) # 2 Doktoren für dieses Beispiel. Beide starten bei 0.append




In [9]:
len(schedule[1])

29

In [5]:
def create_patient_list_from_instance(filepath):
    df = pd.read_json(file_path)

# Auswahl und Anordnung der Spalten mit den Nullen an den exakten Positionen
    instance_patients = np.array([
    [
        row['patient_id'],
        row['arrival_time'],
        row['realized_processing_time'],
        0, 0,row['arrival_time']+row['max_wait_time'], 0, 
        row['weight'],
        row['expected_processing_time']
    ]
    for _, row in df.iterrows()
    ], dtype=int)

    return instance_patients

def calculate_weighted_tardiness(schedule):
    schedule_weighted_tardiness = 0
    for doctor in schedule:
        schedule_weighted_tardiness += nh.weighted_tardiness_per_doc(schedule, doctor)
    return schedule_weighted_tardiness

def check_for_high_urgency(arrived_patients_list):
    # Überprüfen, ob eine 3 oder 4 in der siebten Spalte ist
    return np.any(np.isin(arrived_patients_list[:, 7], [3, 4]))
    


In [22]:
hospital = "italien"
group = 3
tag = 2
file_path = f'{hospital}/group_{group}/gruppe_{group}_{hospital}_{tag}.json.json'
doctor_count = 3
doctor_completion = np.zeros(doctor_count)

waiting_room_schedule = {}
waiting_room_weighted_tardiness = 0
schedule_weighted_tardiness = 0
schedule = {}
deterministic = False
metaheuristic = "VNS"

group_intervals = {
    1: [  0,  15,  30,  45,  60,  75,  90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240],
    2: [240, 255, 270, 285, 300, 315, 330, 345, 360, 375, 390, 405, 420, 435, 450, 465, 480],
    3: [480, 495, 510, 525, 540, 555, 570, 585, 600, 615, 630, 645, 660, 675, 690, 705, 720],
    4: [720, 735, 750, 765, 780, 795, 810, 825, 840, 855, 870, 885, 900, 915, 930, 945, 960],
    5: [ 960,  975,  990, 1005, 1020, 1035, 1050, 1065, 1080, 1095, 1110, 1125, 1140, 1155, 1170, 1185, 1200],
    6: [1200, 1215, 1230, 1245, 1260, 1275, 1290, 1305, 1320, 1335, 1350, 1365, 1380, 1395, 1410, 1425, 1440]
}

instance_patients = create_patient_list_from_instance(file_path)

#metaheuristic = 0
#metaheuristic = "SA"

for i in range(1, doctor_count+1):
    waiting_room_schedule[i] = np.array([], dtype=int).reshape(0, 9)

schedule = deepcopy(waiting_room_schedule)

# Früheste Startzeit der Instanz
start_time = instance_patients[0][1]

# Letzte Startzeit der Instanz
end_time = instance_patients[-1][1]

# Init Array mit doctor release times (expected) T
estimated_treatment_end = np.full(doctor_count, start_time)

# Init Array mit tatsächlichen doctor release times (tatsächlich)
treatment_end = estimated_treatment_end.copy()

time = start_time

while time <= end_time:
    # Array mit Patienten, die zum Zeitpunkt T ankommen
    arrived_patients = instance_patients[:, 1] == time
    arrived_patients_list = instance_patients[arrived_patients]
    high_urgency = False

    if arrived_patients_list.shape[0] > 0:
        #new_patient = True
        #print(f"{time}: Patient arrives:")
        dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, arrived_patients_list, deterministic=deterministic)
        waiting_room_weighted_tardiness = calculate_weighted_tardiness(waiting_room_schedule)
        high_urgency = check_for_high_urgency(arrived_patients_list)
        

        # Metaheuristik
        if metaheuristic == "VNS" and waiting_room_weighted_tardiness != 0 and high_urgency:
            print(f"{time} priority reschedule")
            new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=50, time_limit=60, start_time_array=estimated_treatment_end, deterministic=deterministic)
            if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
                print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}")
                waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
    
    if time in group_intervals[group] and not high_urgency and waiting_room_weighted_tardiness !=0:
        print(f"{time}: reschedule")
        new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=50, time_limit=60, start_time_array=estimated_treatment_end, deterministic=deterministic)
        if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
            print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}")
            waiting_room_schedule = deepcopy(new_waiting_room_schedule)
            waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)

    free_doctors = np.where(treatment_end <= time)[0]
    free_doctors += 1

    if len(free_doctors)>0:
        for doctor in free_doctors:
            start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end, doctor)

    waiting_room_weighted_tardiness = calculate_weighted_tardiness(waiting_room_schedule)
        
    time += 1
    waiting_room_empy = all(array.size == 0 for array in waiting_room_schedule.values())
    if not waiting_room_empy:
        end_time += 1

schedule_weighted_tardiness = calculate_weighted_tardiness(schedule)

print(f"{schedule}, {schedule_weighted_tardiness}")

507 priority reschedule
Bessere Lösung 0.00689697265625: 69 auf 21
507: VNS 69 Verbesserung auf 21
510: reschedule
520 priority reschedule
522 priority reschedule
Bessere Lösung 0.0: 60 auf 6
522: VNS 60 Verbesserung auf 6
525: reschedule
538 priority reschedule
Bessere Lösung 0.0: 54 auf 34
Bessere Lösung 0.1299903392791748: 34 auf 32
Bessere Lösung 0.3619875907897949: 32 auf 29
Bessere Lösung 1.3374645709991455: 29 auf 28
538: VNS 54 Verbesserung auf 28
539 priority reschedule
Bessere Lösung 0.026822566986083984: 151 auf 139
Bessere Lösung 0.06961345672607422: 139 auf 121
Bessere Lösung 0.10503482818603516: 121 auf 111
Bessere Lösung 0.15022706985473633: 111 auf 105
Bessere Lösung 0.20576071739196777: 105 auf 100
Bessere Lösung 0.533778190612793: 100 auf 98
539: VNS 151 Verbesserung auf 98
540: reschedule
552 priority reschedule
Bessere Lösung 0.0: 223 auf 174
Bessere Lösung 0.04864859580993652: 174 auf 168
Bessere Lösung 0.3998544216156006: 168 auf 166
552: VNS 223 Verbesserung auf 

In [8]:
hospital = "italien"
group = 4
tag = 1
file_path = f'{hospital}/group_{group}/gruppe_{group}_{hospital}_{tag}.json.json'
doctor_count = 7
doctor_completion = np.zeros(doctor_count)

waiting_room_schedule = {}
waiting_room_weighted_tardiness = 0
schedule_weighted_tardiness = 0
schedule = {}
deterministic = False
metaheuristic = "VNS"

group_intervals = {
    1: [  0,  15,  30,  45,  60,  75,  90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240],
    2: [240, 255, 270, 285, 300, 315, 330, 345, 360, 375, 390, 405, 420, 435, 450, 465, 480],
    3: [480, 495, 510, 525, 540, 555, 570, 585, 600, 615, 630, 645, 660, 675, 690, 705, 720],
    4: [720, 735, 750, 765, 780, 795, 810, 825, 840, 855, 870, 885, 900, 915, 930, 945, 960],
    5: [ 960,  975,  990, 1005, 1020, 1035, 1050, 1065, 1080, 1095, 1110, 1125, 1140, 1155, 1170, 1185, 1200],
    6: [1200, 1215, 1230, 1245, 1260, 1275, 1290, 1305, 1320, 1335, 1350, 1365, 1380, 1395, 1410, 1425, 1440]
}

instance_patients = create_patient_list_from_instance(file_path)

for i in range(1, doctor_count+1):
    waiting_room_schedule[i] = np.array([], dtype=int).reshape(0, 9)

schedule = deepcopy(waiting_room_schedule)

# Früheste Startzeit der Instanz
start_time = instance_patients[0][1]

# Letzte Startzeit der Instanz
end_time = instance_patients[-1][1]

# Init Array mit doctor release times (expected)
estimated_treatment_end = np.full(doctor_count, start_time)

# Init Array mit tatsächlichen doctor release times (tatsächlich)
treatment_end = estimated_treatment_end.copy()

time = start_time

while time <= end_time:
    # Array mit Patienten, die zum Zeitpunkt T ankommen
    arrived_patients = instance_patients[:, 1] == time
    arrived_patients_list = instance_patients[arrived_patients]
    high_urgency = False

    if arrived_patients_list.shape[0] > 0:
        dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, arrived_patients_list, deterministic=deterministic)
        waiting_room_weighted_tardiness = calculate_weighted_tardiness(waiting_room_schedule)
        high_urgency = check_for_high_urgency(arrived_patients_list)
        

    #     # Metaheuristik
    #     if metaheuristic == "VNS" and waiting_room_weighted_tardiness != 0 and high_urgency:
    #         print(f"{time} priority reschedule")
    #         new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=50, time_limit=60, start_time_array=estimated_treatment_end, deterministic=deterministic)
    #         if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
    #             print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}")
    #             waiting_room_schedule = deepcopy(new_waiting_room_schedule)
    #             waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
    
    # if time in group_intervals[group] and not high_urgency and waiting_room_weighted_tardiness !=0:
    #     print(f"{time}: reschedule")
    #     new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=50, time_limit=60, start_time_array=estimated_treatment_end, deterministic=deterministic)
    #     if new_waiting_room_weighted_tardiness < waiting_room_weighted_tardiness:
    #         print(f"{time}: VNS {waiting_room_weighted_tardiness} Verbesserung auf {new_waiting_room_weighted_tardiness}")
    #         waiting_room_schedule = deepcopy(new_waiting_room_schedule)
    #         waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)


    free_doctors = np.where(treatment_end <= time)[0]
    free_doctors += 1

    if len(free_doctors)>0:
        for doctor in free_doctors:
            start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end, doctor)

    waiting_room_weighted_tardiness = calculate_weighted_tardiness(waiting_room_schedule)
        
    time += 1
    waiting_room_empy = all(array.size == 0 for array in waiting_room_schedule.values())
    if not waiting_room_empy:
        end_time += 1

schedule_weighted_tardiness = calculate_weighted_tardiness(schedule)

print(f"{schedule}, {schedule_weighted_tardiness}")

{1: array([[   0,  729,   15,  729,  744,  789,    0,    2,   17],
       [   8,  747,   19,  747,  766,  762,    0,    3,   21],
       [  15,  763,   22,  766,  788,  778,    0,    3,   21],
       [  22,  781,   21,  788,  809,  841,    0,    2,   17],
       [  27,  817,   34,  817,  851,  832,    0,    3,   21],
       [  35,  849,    6,  851,  857,  864,    0,    3,   21],
       [  42,  896,   16,  896,  912, 1016,    0,    1,   13]]), 2: array([[  1, 731,  17, 731, 748, 791,   0,   2,  17],
       [  9, 752,  14, 752, 766, 872,   0,   1,  13],
       [ 14, 758,  12, 766, 778, 878,   0,   1,  13],
       [ 20, 776,  11, 778, 789, 836,   0,   2,  17],
       [ 26, 814,  10, 814, 824, 874,   0,   2,  17],
       [ 31, 828,  18, 828, 846, 888,   0,   2,  17],
       [ 41, 891,  33, 891, 924, 891,   0,   4,  23]]), 3: array([[  2, 732,  21, 732, 753, 747,   0,   3,  21],
       [ 12, 756,  14, 756, 770, 816,   0,   2,  17],
       [ 17, 772,  10, 772, 782, 832,   0,   2,  17],
     

In [23]:
import numpy as np

array = np.arange(1200, 1441, 15)
#print(array)
array

array([1200, 1215, 1230, 1245, 1260, 1275, 1290, 1305, 1320, 1335, 1350,
       1365, 1380, 1395, 1410, 1425, 1440])

In [24]:
instance_patients

array([[  0,  22,  24,   0,   0,  82,   0,   2,  17],
       [  1,  37,  14,   0,   0,  97,   0,   2,  17],
       [  2,  39,   7,   0,   0,  54,   0,   3,  23],
       [  3,  42,  19,   0,   0,  57,   0,   3,  23],
       [  4,  48,   8,   0,   0, 108,   0,   2,  17],
       [  5,  53,   9,   0,   0, 113,   0,   2,  17],
       [  6,  78,  26,   0,   0, 138,   0,   2,  17],
       [  7,  97,  43,   0,   0, 112,   0,   3,  23],
       [  8, 106,  19,   0,   0, 166,   0,   2,  17],
       [  9, 119,  30,   0,   0, 134,   0,   3,  23],
       [ 10, 128,   4,   0,   0, 248,   0,   1,  11],
       [ 11, 168,  40,   0,   0, 183,   0,   3,  23]])

In [30]:
arrived_patients = instance_patients[:, 1] == 97
arrived_patients_list = instance_patients[arrived_patients]

# Überprüfen, ob eine 3 oder 4 in der siebten Spalte ist
high_urgency = np.any(np.isin(arrived_patients_list[:, 7], [3, 4]))
print(high_urgency)
arrived_patients_list

True


array([[  7,  97,  43,   0,   0, 112,   0,   3,  23]])

In [31]:
group_intervals

NameError: name 'group_intervals' is not defined

In [32]:
group_intervals = {
    1: [  0,  15,  30,  45,  60,  75,  90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240],
    2: [240, 255, 270, 285, 300, 315, 330, 345, 360, 375, 390, 405, 420, 435, 450, 465, 480],
    3: [480, 495, 510, 525, 540, 555, 570, 585, 600, 615, 630, 645, 660, 675, 690, 705, 720],
    4: [720, 735, 750, 765, 780, 795, 810, 825, 840, 855, 870, 885, 900, 915, 930, 945, 960],
    5: [ 960,  975,  990, 1005, 1020, 1035, 1050, 1065, 1080, 1095, 1110, 1125, 1140, 1155, 1170, 1185, 1200],
    6: [1200, 1215, 1230, 1245, 1260, 1275, 1290, 1305, 1320, 1335, 1350, 1365, 1380, 1395, 1410, 1425, 1440]
}

In [36]:
254 in group_intervals[1]

False